# 🎟️ Notebook 2: Sessions vs JWT

Once a user has logged in successfully, how do we *remember* that they are logged in across many requests? HTTP itself is stateless — every request is independent.

There are two popular answers:

1. 🟨 **Server-side sessions** — the server stores everything, and gives the client only a random ID.
2. 🟩 **JSON Web Tokens (JWT)** — the server gives the client a *signed* blob that it can verify on every request without any storage.

Neither is universally "best" — they trade off **revocation**, **size on the wire**, and **statelessness**.

## Learning objectives
- See both schemes work end-to-end, in plain Python.
- Understand the revocation problem with JWT.
- Know when to pick which.

## 🟨 Approach 1: Server-side session

The server keeps a dictionary `session_id -> user_info`. The client only stores the random session id (usually in a cookie). To log a user out, the server just deletes the entry.

In [ ]:
import secrets

# Pretend this is Redis or a database table.
SESSIONS = {}

def login_session(username):
    sid = secrets.token_urlsafe(16)
    SESSIONS[sid] = {"user": username}
    return sid  # send this to the client as a cookie

def whoami_session(sid):
    return SESSIONS.get(sid)

def logout_session(sid):
    SESSIONS.pop(sid, None)

sid = login_session("alice")
print("session id:", sid)
print("whoami:", whoami_session(sid))
logout_session(sid)
print("after logout:", whoami_session(sid))  # None — instantly revoked

assert whoami_session(sid) is None, "logout must revoke the session immediately"

## 🟩 Approach 2: JWT (stateless)

A JWT is just three base64url-encoded JSON parts joined with dots: **header.payload.signature**.

The signature is computed using a secret only the server knows. Anyone can *read* the payload (it is not encrypted!), but only the server can *create* a valid one.

The huge upside: **no server-side storage**. Any server with the secret can verify the token. The downside: you can't easily *revoke* a token before it expires.

In [ ]:
import jwt  # PyJWT
import time

SECRET = "super-secret-only-the-server-knows"

def login_jwt(username, ttl_seconds=60):
    payload = {
        "sub": username,
        "iat": int(time.time()),
        "exp": int(time.time()) + ttl_seconds,
    }
    return jwt.encode(payload, SECRET, algorithm="HS256")

def whoami_jwt(token):
    try:
        return jwt.decode(token, SECRET, algorithms=["HS256"])
    except jwt.PyJWTError as e:
        return f"invalid: {e}"

token = login_jwt("alice", ttl_seconds=2)
print("token:", token)
print("whoami:", whoami_jwt(token))

print("\nWaiting 3 seconds for the token to expire...")
time.sleep(3)
result = whoami_jwt(token)
print("whoami:", result)

# PyJWT checks `exp` for us, but only because we passed it a key AND algorithms.
# An expired token must not authenticate anybody.
assert isinstance(result, str) and "expired" in result, result

In [ ]:
# What if someone tampers with the payload? The signature won't match.
# Use a FRESH token so we are testing the signature check, not the expiry.
import base64, json

def b64url_decode(s: str) -> bytes:
    """Add back the '=' padding that JWT strips. `s + "=="` happens to work for
    most payloads and then mysteriously fails for others — don't do that."""
    return base64.urlsafe_b64decode(s + "=" * (-len(s) % 4))

fresh = login_jwt("alice", ttl_seconds=300)
header, payload, sig = fresh.split(".")

decoded = json.loads(b64url_decode(payload))
print("payload (anyone can read this — JWTs are signed, NOT encrypted):", decoded)
print("header :", json.loads(b64url_decode(header)))

# The attacker edits the payload, trying to become admin.
decoded["sub"] = "admin"
tampered_payload = base64.urlsafe_b64encode(json.dumps(decoded).encode()).decode().rstrip("=")
forged = f"{header}.{tampered_payload}.{sig}"
print("\nforged check:", whoami_jwt(forged))

# Verification must actually run. If this ever returns a dict, the server is
# decoding without verifying — the single most common JWT mistake.
result = whoami_jwt(forged)
assert isinstance(result, str) and "invalid" in result, result
assert whoami_jwt(fresh)["sub"] == "alice", "the untampered token must still work"

print()
print("The signature was computed over the ORIGINAL payload. Once the payload")
print("changes, the signature no longer matches — the server rejects the token.")

## ❌ The JWT revocation problem — and a deny-list fix

A valid JWT is valid until it expires. If Alice's laptop gets stolen 30 minutes into a 24-hour token, the thief stays logged in for another 23.5 hours. That is the price of statelessness.

A common workaround is a **deny-list** (a.k.a. block-list or revocation list): a small store of token IDs that were logged out before their natural expiry. Every request checks it. Yes, this re-introduces a tiny bit of state — but only for *revoked* tokens, not every session.

In [ ]:
# Give each token a unique id ("jti") so we can revoke individual tokens.
import jwt, time

REVOKED = set()  # store of revoked jti values

def login_jwt_v2(username, ttl_seconds=60):
    payload = {
        "sub": username,
        "jti": secrets.token_urlsafe(8),
        "iat": int(time.time()),
        "exp": int(time.time()) + ttl_seconds,
    }
    return jwt.encode(payload, SECRET, algorithm="HS256")

def revoke(token):
    data = jwt.decode(token, SECRET, algorithms=["HS256"])
    REVOKED.add(data["jti"])

def whoami_jwt_v2(token):
    try:
        data = jwt.decode(token, SECRET, algorithms=["HS256"])
    except jwt.PyJWTError as e:
        return f"invalid: {e}"
    if data["jti"] in REVOKED:
        return "invalid: revoked"
    return data

t = login_jwt_v2("alice", ttl_seconds=300)
print("before revoke:", whoami_jwt_v2(t))
revoke(t)
print("after revoke :", whoami_jwt_v2(t))

assert whoami_jwt_v2(t) == "invalid: revoked"
# ...and note what the deny-list did NOT do: the token is still cryptographically
# valid. Anything that verifies it without consulting REVOKED still accepts it.
assert jwt.decode(t, SECRET, algorithms=["HS256"])["sub"] == "alice"
print("\n⚠️  The revoked token still passes signature+expiry verification.")
print("    Every service that trusts this JWT must check the deny-list too, or")
print("    you have revocation on one service and not the others.")

## 🍪 Where the token lives, and how it gets attacked

A JWT is only as safe as the two things around it: **how you verify it** and **where
the browser keeps it**. The next two sections attack the verification with the two
classic JWT exploits; after that we come back to storage.

## 💣 Attack 1: `alg: none`

The JWT header carries the algorithm the token *claims* was used. Early libraries
trusted it, and the spec includes an algorithm called `none` meaning "unsigned".
Put the two together and an attacker can hand you a token with **no signature at
all** and an arbitrary payload.

The defence is one line and it is non-negotiable: **the server decides the algorithm,
never the token**. Below we forge an `alg: none` admin token and try it against two
verifiers — a permissive one and ours.

In [ ]:
import base64, json, jwt

def b64url(raw: bytes) -> str:
    return base64.urlsafe_b64encode(raw).rstrip(b"=").decode()

# An unsigned token: header says "none", signature segment is empty.
evil_header  = b64url(json.dumps({"alg": "none", "typ": "JWT"}).encode())
evil_payload = b64url(json.dumps({"sub": "admin", "role": "superuser"}).encode())
alg_none_token = f"{evil_header}.{evil_payload}."

print("forged token:", alg_none_token)
print("(note the trailing dot — there is no signature)\n")

# ❌ A permissive verifier. People write this to "debug" and it ships.
naive = jwt.decode(alg_none_token, "", algorithms=["none"],
                   options={"verify_signature": False})
print("❌ permissive verifier says:", naive)
assert naive["sub"] == "admin", "the attack should succeed against a naive verifier"

# ✅ Our verifier pins the algorithm list, so the token never gets that far.
try:
    jwt.decode(alg_none_token, SECRET, algorithms=["HS256"])
    raise AssertionError("alg:none must never be accepted")
except jwt.InvalidAlgorithmError as e:
    print("✅ strict verifier rejects it:", e)

## 💣 Attack 2: algorithm confusion (RS256 → HS256)

Bigger systems sign with an **asymmetric** key: the auth server holds an RSA private
key, every other service verifies with the matching **public** key. The public key is,
by design, public — it is published at a `/.well-known/jwks.json` endpoint.

Here is the trick. `HS256` is symmetric: the *same* value both signs and verifies. So
if a server will accept **either** `RS256` **or** `HS256`, an attacker can:

1. Fetch the public key (it's public).
2. Sign their own payload with `HS256`, using **the public key bytes as the HMAC secret**.
3. Send it. The server sees `alg: HS256`, reaches for "the key" — the public key — and
   the signature checks out.

The attacker just forged a token using nothing but public information.

In [ ]:
import hmac, hashlib
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import serialization

# The auth server's key pair. Only `public_pem` ever leaves the building.
_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
private_pem = _key.private_bytes(
    serialization.Encoding.PEM,
    serialization.PrivateFormat.PKCS8,
    serialization.NoEncryption()).decode()
public_pem = _key.public_key().public_bytes(
    serialization.Encoding.PEM,
    serialization.PublicFormat.SubjectPublicKeyInfo).decode()

legit = jwt.encode({"sub": "alice"}, private_pem, algorithm="RS256")
assert jwt.decode(legit, public_pem, algorithms=["RS256"])["sub"] == "alice"
print("legit RS256 token verifies ✔")

# --- the forgery, built by hand from public information only ---
hdr = b64url(json.dumps({"alg": "HS256", "typ": "JWT"}, separators=(",", ":")).encode())
pld = b64url(json.dumps({"sub": "admin"}, separators=(",", ":")).encode())
signing_input = f"{hdr}.{pld}".encode()
sig = b64url(hmac.new(public_pem.encode(), signing_input, hashlib.sha256).digest())
confused_token = f"{hdr}.{pld}.{sig}"
print("forged with the PUBLIC key:", confused_token[:48], "...")

In [ ]:
# ❌ A hand-rolled verifier that "supports both algorithms" — the vulnerable pattern.
def vulnerable_verify(token: str, key: str):
    hdr_b64, pld_b64, sig_b64 = token.split(".")
    alg = json.loads(b64url_decode(hdr_b64))["alg"]        # 🚩 trusting the token
    if alg == "HS256":
        expected = b64url(hmac.new(key.encode(),
                                   f"{hdr_b64}.{pld_b64}".encode(),
                                   hashlib.sha256).digest())
        if hmac.compare_digest(expected, sig_b64):
            return json.loads(b64url_decode(pld_b64))
        raise ValueError("bad signature")
    return jwt.decode(token, key, algorithms=["RS256"])

owned = vulnerable_verify(confused_token, public_pem)
print("❌ vulnerable verifier says:", owned)
assert owned["sub"] == "admin", "the confusion attack should succeed here"

# ✅ Pin a single algorithm. The header's claim is now irrelevant.
try:
    jwt.decode(confused_token, public_pem, algorithms=["RS256"])
    raise AssertionError("algorithm confusion must be rejected")
except jwt.InvalidAlgorithmError as e:
    print("✅ pinning algorithms=['RS256'] rejects it:", e)

# ✅ Modern PyJWT adds a second layer: it refuses to use an RSA key as an HMAC
# secret at all, even if you ask for HS256. Older libraries and other language
# ecosystems do not, which is why pinning the algorithm is the real defence.
try:
    jwt.decode(confused_token, public_pem, algorithms=["HS256", "RS256"])
    raise AssertionError("should not have verified")
except jwt.InvalidKeyError as e:
    print("✅ PyJWT also refuses the key type:", e)

### 🔐 The JWT verification checklist

Everything above collapses into a short list. On **every** request, on **every**
service:

1. **Pin the algorithms** — `algorithms=["RS256"]`, a list you control. Never read
   `alg` from the token.
2. **Verify the signature.** `jwt.decode(..., options={"verify_signature": False})`
   belongs in a debugger, never on a request path.
3. **Check `exp`** (and `nbf`). PyJWT does this by default; some libraries don't.
4. **Check `aud` and `iss`.** A perfectly valid token issued for a *different*
   audience is not valid for you — this is how one tenant's token gets replayed at
   another tenant's API.
5. **Remember it is not encrypted.** Base64 is not encryption; anyone holding the
   token can read every claim. No PII, no internal ids you wouldn't put in a URL.
6. **Keep them short-lived**, because you cannot un-issue one (see the deny-list above).

### 🍪 Where the token lives

- **`HttpOnly`** — JavaScript can't read the cookie, so XSS can't steal it via
  `document.cookie`. It does **not** stop CSRF on its own.
- **`Secure`** — only sent over HTTPS.
- **`SameSite=Lax` or `Strict`** — the browser won't attach the cookie to most
  cross-site requests. This is the real CSRF defence.

For JWTs kept in `localStorage` in an SPA you lose all three: any XSS is full token
theft. Prefer cookies with these flags when you can.

### HS256 vs RS256

Our demo uses a single shared secret (`HS256`), which means every service that can
*verify* a token can also *mint* one. With `RS256`/`ES256` the auth server holds the
private key and everyone else gets only the public key — a much better blast radius,
and the reason it is the norm in microservice fleets.

## 🤔 Which to pick?

| Concern | Sessions | JWT |
|---|---|---|
| Revocation (force logout) | ✅ instant | ❌ must wait for expiry (or maintain a deny-list) |
| Server storage | ❌ needs Redis/DB | ✅ stateless |
| Size on the wire | ✅ small (just an id) | ❌ bigger (JSON payload) |
| Cross-service auth | meh — every service hits the session store | ✅ each service verifies with the public key |

**Rule of thumb:** classic web app with one backend → sessions are simpler. Microservices, mobile, or third-party APIs → JWT (often *short-lived* JWT + a long-lived refresh token).